In [1]:
from api_helpers.uis_api import *
from api_helpers.world_bank_api import *

import pandas as pd
import requests

from pathlib import Path
import json
import sys
sys.path.append(r"C:\Users\F.Turner\Documents\00. Analyses")
import use_funcs
from use_funcs import find_project_root

In [2]:
API_KEY = "51addd610c2d4725806a8a49c1eaab8f"

In [3]:
PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "path_config.json"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PATHS = json.load(f)

IMP_DIR = (PROJECT_ROOT / PATHS["imports_dir"]).resolve()
FSI_DIR = (PROJECT_ROOT / PATHS["fsi_dir"]).resolve()
HO_DIR = (PROJECT_ROOT / PATHS["handoff_dir"]).resolve()
EXP_DIR = (PROJECT_ROOT / PATHS["exports_dir"]).resolve()
FIG_DIR = (PROJECT_ROOT / PATHS["figures_dir"]).resolve()
TAB_DIR = (PROJECT_ROOT / PATHS["tables_dir"]).resolve()

for folder in [IMP_DIR, FSI_DIR, HO_DIR, EXP_DIR, FIG_DIR, TAB_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMP_DIR:", IMP_DIR)
print("FSI_DIR:", FSI_DIR)
print("HO_DIR:", HO_DIR)
print("EXP_DIR:", EXP_DIR)
print("FIG_DIR:", FIG_DIR)
print("TAB_DIR:", TAB_DIR)

PROJECT_ROOT: C:\Users\F.Turner\Documents\00. Analyses\Education Financing
IMP_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Data
FSI_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Data\FSI Raw Data
HO_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Handoff
EXP_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Results
FIG_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Results\figures
TAB_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Results\tables


In [4]:
iso3 = pd.read_csv(IMP_DIR / "iso3.csv")

wb_countries_url = "https://api.worldbank.org/v2/country?format=json&per_page=400"
wb_response = requests.get(wb_countries_url, timeout=30)
wb_response.raise_for_status()
wb_countries = wb_response.json()[1]

wb_country_meta = pd.DataFrame(
    [
        {
            "iso3": c.get("id"),
            "wb_income_group": (c.get("incomeLevel") or {}).get("value"),
        }
        for c in wb_countries
    ]
)

iso3 = iso3.drop(columns=["wb_income_group"], errors="ignore")
iso3 = iso3.merge(wb_country_meta, on="iso3", how="left")
iso3.to_csv(IMP_DIR / "iso3.csv", index=False)

In [5]:
iso_list = iso3["iso3"].tolist()

## **Import Control Data**

## 01. Education System Data (UIS Data)

In [6]:
# Import data using the UIS API helper function, 
# for indicator "School Aged Population Primary - Total (SAP.1)", 
# for the years 2010 to 2026.

sap = uis_get(indicators='SAP.1', entities=iso_list, start_year=2010, end_year=2026)

In [7]:
pop_total = wb_get(indicators='SP.POP.TOTL',
       countries='All', 
        source=2, 
        start_year=2010, 
        end_year=2026, 
        format='long', 
        drop_missing=False)

In [8]:
# Build SAP control with total population denominator and ISO3 coverage checks.
sap = sap.copy()
sap = sap.rename(columns={"entity_id": "iso3", "value": "sap"})

pop_total_clean = pop_total.rename(
    columns={"country_id": "iso3", "value": "pop_total"}
    )[["iso3", "year", "pop_total"]]

sap = sap.merge(pop_total_clean, on=["iso3", "year"], how="left")
sap["sap_share"] = sap["sap"] / sap["pop_total"]

sap_iso3 = set(sap["iso3"].dropna().unique())
country_iso3 = set(iso3["iso3"].dropna().unique())
missing_iso3 = sorted(sap_iso3 - country_iso3)

In [9]:
priv_ex = uis_get(indicators=['ETOIP.1.PR', 'XGDP.FSHH.FFNTR'], 
                  start_year=2010, end_year=2026)

In [10]:
map = {'ETOIP.1.PR': 'private_enrolment_pct', 
       'XGDP.FSHH.FFNTR': 'hh_exp_pct'}

priv_ex = priv_ex.rename(columns={"entity_id": "iso3", "indicator_id": "indicator"})

priv_ex["indicator"] = priv_ex["indicator"].map(map)

priv_ex = priv_ex.pivot(index=["iso3", "year"], columns="indicator", values="value").reset_index().rename_axis(columns=None)

In [11]:
uis_controls = pd.merge(sap[["iso3", "year", "sap", "pop_total", "sap_share"]], priv_ex, on=["iso3", "year"], how="outer")

In [12]:
uis_controls.to_csv(IMP_DIR / "uis_controls.csv", index=False)

## 02. State Fragility (Institute for Peace Data)

In [13]:
def fsi_data(year):
    fragility = pd.read_excel(FSI_DIR / f"fsi-{year}.xlsx")

    # Standardize key join field.
    fragility = fragility.rename(columns={"Country": "country_name"})
    fragility["country_name"] = fragility["country_name"].astype(str).str.strip()

    # Keep ISO3 lookup unique before joining.
    iso_lookup = iso3[["iso3", "country_name"]].drop_duplicates(subset="country_name", keep="first")
    iso_lookup["country_name"] = iso_lookup["country_name"].astype(str).str.strip()
    fsi = pd.merge(iso_lookup, fragility, on="country_name", how="right")
    

    # Manual harmonization for known country-name variants.
    fsi_iso3_map = {
        "Bahamas": "BHS",
        "Cape Verde": "CPV",
        "Congo Democratic Republic": "COD",
        "Congo Republic": "COG",
        "Czech Republic": "CZE",
        "Egypt": "EGY",
        "Gambia": "GMB",
        "Guinea Bissau": "GNB",
        "Iran": "IRN",
        "Israel and West Bank": "ISR",
        "Kyrgyzstan": "KGZ",
        "Laos": "LAO",
        "Macedonia": "MKD",
        "Micronesia": "FSM",
        "North Korea": "PRK",
        "Palestine": "PSE",
        "Russia": "RUS",
        "Slovakia": "SVK",
        "Somalia": "SOM",
        "South Korea": "KOR",
        "Swaziland": "SWZ",
        "Syria": "SYR",
        "Turkey": "TUR",
        "Uruguay": "URY",
        "Venezuela": "VEN",
        "Vietnam": "VNM",
        "Yemen": "YEM",
        "Côte d'Ivoire": "CIV",
    }

    fsi["iso3"] = fsi["iso3"].fillna(fsi["country_name"].map(fsi_iso3_map))
    fsi["year"] = year
    fsi = fsi.drop(columns=["Year", "country_name"])

    return fsi

In [14]:
fsi_long = pd.concat(
    [fsi_data(year) for year in range(2011, 2024)],
    ignore_index=True,
)

fsi_long.columns = (
    fsi_long.columns.str.strip()
    .str.lower()
    .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
    .str.replace(r"_+", "_", regex=True)
    .str.replace(r"(^_+|_+$)", "", regex=True)
)

fsi_long = fsi_long.sort_values(by=["iso3", "year"]).reset_index(drop=True)

fsi = fsi_long.rename(columns={'total': 'fsi_total'})


In [15]:
fsi_summary = fsi[['iso3', 'year', 'fsi_total']]

In [16]:

fsi.to_csv(IMP_DIR / "fsi.csv", index=False)
fsi_summary.to_csv(IMP_DIR / "fsi_summary.csv", index=False)


In [17]:
# Add Column on Linguistic Fractionalisation
fractionalisation_2000 = pd.read_csv(
    IMP_DIR / "fractionalisation_2000.csv",
    sep=";",
    decimal=",",
    encoding="utf-8",
)

lang_frac = fractionalisation_2000[["ccodealp", "year", "al_language2000"]].rename(
    columns={"ccodealp": "iso3", "al_language2000": "lang_frac"}
)

lang_frac.to_csv(IMP_DIR / "lang_frac.csv", index=False)

## 03. Macro-economic Controls (WB Data)

In [18]:

controls = wb_get(
    countries='all',
    indicators=[
        "NY.GDP.PCAP.KD",
        "NY.GDP.MKTP.KD.ZG",
        "FP.CPI.TOTL.ZG",
        "GC.DOD.TOTL.GD.ZS",
    ],
    source=2,
    start_year=2010,
    end_year=2026,
    format="long",
    drop_missing=True,
)

WorldBankApiError: Failed to retrieve data from the World Bank API.

In [ ]:
macro = controls.drop(columns=["country_api_id",
                               'country_name',
                               "source_id", 
                               "unit", 
                               "decimal", 
                               'obs_status', 
                               'last_updated'])

macro = macro.pivot_table(index=["country_id", "year"], 
                          columns="indicator_name", values="value").reset_index().rename_axis(columns=None)


In [ ]:
macro = macro.rename(columns={
    'country_id' : 'iso3',
    'Central government debt, total (% of GDP)' : 'debt_pct_gdp',
    'GDP growth (annual %)' : 'gdp_growth_pct',
    'GDP per capita (constant 2015 US$)' : 'gdppc_2015_usd',
    'Inflation, consumer prices (annual %)' : 'inflation_pct',
})

income_lookup = (
    iso3[["iso3", "wb_income_group"]]
    .drop_duplicates(subset=["iso3"])
    .rename(columns={"wb_income_group": "income_group"})
)
macro = macro.merge(income_lookup, on="iso3", how="left")

In [ ]:
macro.to_csv(IMP_DIR / "macro.csv", index=False)